# dieWolke on Colab: runbook

Run the sections in order. Each training cell is resumable: if the session dies, run the setup
cells again and re-run the same training cell, and it continues from the last checkpoint on Drive.

| Section | When | Rough time on a T4 (estimates, measure with section 4) |
|---|---|---|
| 1-2 Setup | every session | 2 min |
| 3 Data | download once; extract + cache every session | download: depends on the server; extract 30-60 min; cache 10-25 min |
| 4 Timing check | once | 5 min |
| 5 Masker + mask ablation | once | 30-90 min |
| 6 Full model and ablations | one run per session or two | several hours each; use `max_minutes` |
| 7 Evaluation | after training | 10-30 min |
| 8 Baselines (published weights + XGBoost) | once | 30-60 min |
| 9 Kharif benchmark | once (needs an Earth Engine project) | 1-3 h, mostly downloads |

Runtime: **GPU (T4)**. Keep this tab active; free Colab disconnects idle sessions.

## 1. Session setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/dieWolke'          # runs, results and the code copy live here
RAW = '/content/drive/MyDrive/sen12ms_cr'          # SEN12MS-CR .tar.gz archives (downloaded once)
DATA = '/content/data/sen12ms_cr'                  # extracted tifs (local disk, lost on disconnect)
CACHE = '/content/cache'                           # .npy caches (local disk, ~4 MB per patch)
RUNS = f'{DRIVE}/runs'
os.makedirs(RUNS, exist_ok=True)
# Every training/eval command below gets these paths:
PATHS = f'data.root={DATA} data.cache={CACHE} train.out_dir={RUNS}'

## 2. Get the code
Either clone the repository (if this code is pushed) or copy the folder to Drive once and copy it
from there.

In [ ]:
REPO = 'https://github.com/Rasvanth28/DieWolke.git'
if not os.path.exists('/content/DieWolke'):
    !git clone -q {REPO} /content/DieWolke || cp -r {DRIVE}/code/DieWolke /content/DieWolke
%cd /content/DieWolke
!pip install -q -e ".[clouds]"
!python -m pytest -q tests                        # ~1 min on CPU/GPU; everything should pass

## 3. Data: SEN12MS-CR

**Download once.** Paste the archive folder URL from the SEN12MS-CR page on mediaTUM
(https://mediatum.ub.tum.de/1554803) into `SEN12MSCR_URL`. It is read from the environment so it
never ends up in a file. Spring only by default; add `--seasons spring summer fall winter` for all.

In [ ]:
os.environ['SEN12MSCR_URL'] = ''   # paste the URL here for this session only; do not commit it
if os.environ['SEN12MSCR_URL']:
    !python scripts/prepare_sen12mscr.py download --drive {RAW}

**Every session:** extract to local disk and build the caches (train/val/test).

In [ ]:
!python scripts/prepare_sen12mscr.py extract --drive {RAW} --local {DATA}
!python scripts/prepare_sen12mscr.py cache --config configs/full.yaml --set {PATHS}
!python scripts/visualize_samples.py --cache {CACHE} --split test --n 4 --out /content/samples.png
from IPython.display import Image; Image('/content/samples.png')

## 4. Timing check
200 steps of the full model into a scratch folder. `sec_per_step` x 60000 is the full run; set
`train.steps` or `train.max_minutes` accordingly. If it is slow because of disk reads rather than
the GPU, lower `data.n_train` so the cache fits in RAM.

In [ ]:
!python -m diewolke.train --config configs/full.yaml --set {PATHS} name=timing train.steps=200 train.val_every=200 train.log_every=50 train.out_dir=/content/scratch_runs

## 5. Stage 1: masker, and the mask ablation (C5)
`experiments/mask_ablation.py` is the quick U-Net version of C5. `two_stage` / `end_to_end` in
section 6 repeat it with the real bridge model.

In [ ]:
!python -m diewolke.train --config configs/masker.yaml --set {PATHS}

In [ ]:
!python experiments/mask_ablation.py --data_root {DATA} --cache /content/cache_ablation --out {DRIVE}/mask_ablation

## 6. Stage 2: the full model and its ablations
One cell per run. `train.max_minutes=200` stops cleanly with a checkpoint before free-tier
limits; re-run the cell in the next session to continue. Order of priority:
1. `full` and `matched_baseline` (the main comparison)
2. `no_bga`, `no_ndvi`, `no_unc` (one contribution removed each)
3. `two_stage`, `end_to_end` (C5 with the real model; `two_stage` needs the masker from section 5)

In [ ]:
!python -m diewolke.train --config configs/full.yaml --set {PATHS} train.max_minutes=200

In [ ]:
!python -m diewolke.train --config configs/matched_baseline.yaml --set {PATHS} train.max_minutes=200

In [ ]:
!python -m diewolke.train --config configs/no_bga.yaml --set {PATHS} train.max_minutes=200

In [ ]:
!python -m diewolke.train --config configs/no_ndvi.yaml --set {PATHS} train.max_minutes=200

In [ ]:
!python -m diewolke.train --config configs/no_unc.yaml --set {PATHS} train.max_minutes=200

In [ ]:
!python -m diewolke.train --config configs/two_stage.yaml --set {PATHS} train.max_minutes=200 model.masker_ckpt={RUNS}/masker

In [ ]:
!python -m diewolke.train --config configs/end_to_end.yaml --set {PATHS} train.max_minutes=200

## 7. Evaluation (Stage 4)
Real clouds (the standard SEN12MS-CR protocol, comparable with DB-CR Table I only when trained and
tested on the same seasons) and synthetic clouds (exact ground truth). The first run is the
reference for paired differences. `--save_predictions` keeps reconstructions for figures and the
crop task.

In [ ]:
EVAL = f'{DRIVE}/eval'
RUN_LIST = ' '.join(f'{RUNS}/{n}' for n in ['full', 'matched_baseline', 'no_bga', 'no_ndvi', 'no_unc', 'end_to_end']
                    if os.path.exists(f'{RUNS}/{n}/best.pt'))
!python -m diewolke.evaluate --runs {RUN_LIST} --cache {CACHE} --split test --mode real --out {EVAL}/test_real --save_predictions
!python -m diewolke.evaluate --runs {RUN_LIST} --cache {CACHE} --split test --mode synthetic --out {EVAL}/test_synth
if os.path.exists(f'{RUNS}/two_stage/best.pt'):
    !python -m diewolke.evaluate --runs {RUNS}/two_stage --masker {RUNS}/masker --cache {CACHE} --split test --mode synthetic --out {EVAL}/test_synth_two_stage
print(open(f'{EVAL}/test_real/report.md').read())

In [ ]:
# more bridge steps, and stochastic samples for the sample-spread uncertainty
!python -m diewolke.evaluate --runs {RUNS}/full --cache {CACHE} --split test --mode real --steps 5 --eta 1.0 --n_samples 4 --out {EVAL}/test_real_5steps
!python -m diewolke.compare {EVAL}/test_real/per_image_full.csv {EVAL}/test_real/per_image_matched_baseline.csv --per_band
!python scripts/visualize_samples.py --cache {CACHE} --split test --pred {EVAL}/test_real/pred_full.npy --scale {EVAL}/test_real/scale_full.npy --n 4 --out /content/recon.png
Image('/content/recon.png')

## 8. Baselines
Published models from their authors' weights (inference only; see `baselines/external.md` for
which UnCRtainTS checkpoint to pick) and the classical XGBoost SAR-regression gap filling. The
DSen2-CR re-implementation (`configs/dsen2cr.yaml`) is optional now.

In [ ]:
!pip install -q fvcore timm
!git clone -q https://github.com/PatrickTUM/UnCRtainTS /content/UnCRtainTS
!git clone -q https://github.com/xufangchn/GLF-CR /content/GLF-CR
UNCR_WEIGHTS = f'{DRIVE}/uncrtaints_ckpt'   # downloaded from the pCloud link in the UnCRtainTS README
UNCR_EXP = ''                               # a checkpoint folder whose conf.json has "pretrain": true
GLF_CKPT = f'{DRIVE}/glfcr/CR_net.pth'      # from the Google Drive link in the GLF-CR README
!python baselines/run_external.py uncrtaints --repo /content/UnCRtainTS --weights {UNCR_WEIGHTS} --experiment {UNCR_EXP} --cache {CACHE} --split test --mode real --out {EVAL}/pred_uncrtaints_real.npy
!python baselines/run_external.py glfcr --repo /content/GLF-CR --ckpt {GLF_CKPT} --cache {CACHE} --split test --mode real --out {EVAL}/pred_glfcr_real.npy
!python baselines/xgb_reconstruct.py --cache {CACHE} --split test --mode real --out {EVAL}/pred_xgb_real.npy
!python -m diewolke.evaluate --runs {RUNS}/full {RUNS}/matched_baseline --preds uncrtaints={EVAL}/pred_uncrtaints_real.npy,{EVAL}/scale_uncrtaints_real.npy glfcr={EVAL}/pred_glfcr_real.npy xgb={EVAL}/pred_xgb_real.npy --cache {CACHE} --split test --mode real --out {EVAL}/test_real_baselines

## 9. Indian Kharif benchmark (C4)
Needs a Google Cloud project with the Earth Engine API enabled. Edit `kharif/kharif.yaml`
(project, districts) first. `find_scenes.py` is also the early feasibility check for the
"rare clear Kharif scenes" risk.

In [ ]:
!pip install -q earthengine-api pyproj
import ee; ee.Authenticate()
!python kharif/find_scenes.py --config kharif/kharif.yaml
!python kharif/export_patches.py --config kharif/kharif.yaml --dry_run

In [ ]:
!python kharif/export_patches.py --config kharif/kharif.yaml
!python kharif/build_benchmark.py --config kharif/kharif.yaml

In [ ]:
BENCH = f'{DRIVE}/kharif_benchmark'
!python -m diewolke.evaluate --runs {RUNS}/full {RUNS}/matched_baseline --cache {BENCH} --split kharif_synth --calibrate_on "" --out {EVAL}/kharif_synth --save_predictions
!python baselines/xgb_reconstruct.py --cache {BENCH} --split kharif_synth --train_cache {CACHE} --train_split train --out {EVAL}/kharif_synth/pred_xgb.npy
!python kharif/crop_classification.py --cache {BENCH} --split kharif_synth --out {EVAL}/kharif_crops \
    --preds full={EVAL}/kharif_synth/pred_full.npy,{EVAL}/kharif_synth/scale_full.npy matched={EVAL}/kharif_synth/pred_matched_baseline.npy xgb={EVAL}/kharif_synth/pred_xgb.npy